In [1]:
import pandas as pd
import sqlite3

#read csv using pandas
df_telco = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

#connect to sql
conn = sqlite3.connect("telco-customer")
df_telco.to_sql('churn_table', conn, index=False, if_exists='replace')

query1 = """
    SELECT 
    COUNT(customerID) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS total_churn,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(customerID) * 100, 2) AS churn_rate_percentage
    FROM churn_table;
"""
df_data = pd.read_sql_query(query1, conn)
df_data



,total_customers,total_churn,churn_rate_percentage
0,7043,1869,26.54


In [2]:
query2 = """
    SELECT  
    CASE 
        WHEN tenure < 12 THEN ' <1 tahun' 
        WHEN tenure <= 24 THEN ' 1 - 2 tahun'
        ELSE '> 2 tahun'
    END AS tenure_group,
    COUNT(customerID) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS total_churn,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(customerID) * 100, 2) AS churn_rate_percentage
    FROM churn_table
    GROUP BY tenure_group
    
"""
df_data2 = pd.read_sql_query(query2, conn)
df_data2

,tenure_group,total_customers,total_churn,churn_rate_percentage
0,1 - 2 tahun,1141,332,29.10
1,<1 tahun,2069,999,48.28
2,> 2 tahun,3833,538,14.04


In [3]:
query3 = """
    SELECT
        contract,
        COUNT(customerID),
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(customerID) * 100, 2)  AS Churn_Rate_Percentage,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN MonthlyCharges END), 2) AS 'Average Monthly Churn',
    ROUND(AVG(CASE WHEN Churn = 'No' THEN MonthlyCharges END), 2) AS 'Average Monthly non Churn'
    FROM churn_table
    GROUP BY Contract
"""
df_data3 = pd.read_sql_query(query3,conn)
df_data3

,Contract,COUNT(customerID),Churn_Rate_Percentage,Average Monthly Churn,Average Monthly non Churn
0,Month-to-month,3875,42.71,73.02,61.46
1,One year,1473,11.27,85.05,62.51
2,Two year,1695,2.83,86.78,60.01


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split


df_raw = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce').fillna(0)
df_raw['Churn'] = df_raw['Churn'].map({'Yes': 1, 'No': 0})
df_clean = df_raw.drop(columns=['customerID'])
df_clean = pd.get_dummies(df_clean, drop_first=True)


X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_absolute_error

#Initializing random forest model
model_rf = RandomForestClassifier(random_state=42)

#Fitting the model
model_rf.fit(X_train, y_train)

#Predicting the data
y_pred = model_rf.predict(X_test)

#Evaluating the output
print("Model accuration:", accuracy_score(y_test, y_pred))
print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, y_pred))

Akurasi Model: 0.7849538679914834
Mean Absolute Error (MAE): 0.2150461320085167


In [6]:
# Pick 1 customer from test set as a demo
sample_customer = X_test.iloc[[0]]

# Predict using Random Forest model
prediction = model_rf.predict(sample_customer)[0]

print("=== New Customer Prediction ===")
if prediction == 1:
    print("Status: CHURN (Customer is likely to cancel subscription)")
else:
    print("Status: Save (Customer is likely to stay) ")

=== New Customer Prediction ===
Status: Save (Customer is likely to stay) 
